# SQL + Data Analysis Agent

In this notebook, we build an AI agent that converts natural language questions into SQL queries, executes them safely against a database, and interprets the results in plain English. This text-to-SQL pattern is one of the most practical agent architectures for business intelligence and data analysis.

## Architecture Overview

The SQL Data Analysis Agent follows a structured workflow to turn questions into answers:

1. **Receive a question** - The user asks a natural language question about their data (e.g., "What are our top-selling products?")
2. **Inspect the schema** - The agent examines the database structure (tables, columns, types) so it knows what data is available
3. **Generate SQL** - Using its understanding of the schema, the agent writes a SQL query (text-to-SQL)
4. **Validate the query** - Before execution, safety checks reject any destructive operations and enforce row limits
5. **Execute and return results** - The validated query runs against the database and results are returned
6. **Interpret results** - The agent reads the raw data and provides a clear, plain-English answer with insights

```
                    SQL Data Analysis Agent
                    =======================

  User Question          Agent (LLM)              SQLite Database
  ─────────────          ───────────              ───────────────
       │                      │                         │
       │  "Top customers?"    │                         │
       │─────────────────────>│                         │
       │                      │   inspect_schema()      │
       │                      │────────────────────────>│
       │                      │   <schema details>      │
       │                      │<────────────────────────│
       │                      │                         │
       │                      │   validate + execute    │
       │                      │   SELECT ... FROM ...   │
       │                      │────────────────────────>│
       │                      │   <query results>       │
       │                      │<────────────────────────│
       │                      │                         │
       │  Plain-English       │                         │
       │  answer + insights   │                         │
       │<─────────────────────│                         │
```

The key advantage of this architecture is that the agent can **reason about the schema** before writing SQL, which dramatically reduces errors like referencing non-existent columns.

## Safety Considerations

Text-to-SQL agents interact directly with databases, which makes **safety guardrails absolutely critical**. Without them, a model could inadvertently (or through prompt injection) execute destructive queries.

Our agent implements several layers of protection:

| Guardrail | Purpose |
|-----------|--------|
| **SQL validation** | Reject any `DROP`, `DELETE`, `UPDATE`, `INSERT`, `ALTER`, `CREATE`, or `TRUNCATE` statements before they reach the database |
| **Row limits** | Automatically append `LIMIT` clauses to prevent queries that return millions of rows and exhaust memory |
| **Read-only access** | The agent is explicitly designed for `SELECT` queries only |
| **Schema inspection** | By giving the model access to the real schema, we reduce hallucinated table/column names that would cause errors |

In a production environment, you would also want:
- A read-only database user/connection
- Query execution timeouts
- Query complexity limits (e.g., no more than N joins)
- Logging and auditing of all executed queries

## Setup

In [ ]:
# DEPENDENCY: pip install -q openai
# (packages should be pre-installed in venv before running this script)

In [ ]:
import os
import json
import sqlite3
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")

client = OpenAI()
MODEL = "gpt-4o"
MAX_ROW_LIMIT = 50  # Safety: limit result rows

## Create Sample Database

We will create an in-memory SQLite database representing a small e-commerce company. It has four tables:

- **customers** - Who is buying (name, location, lifetime spend)
- **products** - What is for sale (name, category, price, stock)
- **orders** - Each purchase transaction
- **order_items** - Individual line items within each order

The sample data is designed to produce interesting analytical results with varied countries, product categories, date ranges, and spending patterns.

In [ ]:
# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Create tables
conn.executescript("""
CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL,
    city TEXT NOT NULL,
    country TEXT NOT NULL,
    signup_date TEXT NOT NULL,
    lifetime_spend REAL NOT NULL
);

CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL,
    stock_quantity INTEGER NOT NULL
);

CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    total_amount REAL NOT NULL,
    status TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(id)
);

CREATE TABLE order_items (
    id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    FOREIGN KEY (order_id) REFERENCES orders(id),
    FOREIGN KEY (product_id) REFERENCES products(id)
);
""")

# Insert customers (10 customers across various countries)
customers = [
    (1, 'Alice Johnson', 'alice@example.com', 'New York', 'USA', '2023-01-15', 2450.00),
    (2, 'Bob Smith', 'bob@example.com', 'London', 'UK', '2023-03-22', 1870.50),
    (3, 'Carla Mendes', 'carla@example.com', 'Sao Paulo', 'Brazil', '2023-05-10', 3200.75),
    (4, 'David Kim', 'david@example.com', 'Seoul', 'South Korea', '2023-06-01', 980.00),
    (5, 'Elena Rossi', 'elena@example.com', 'Milan', 'Italy', '2023-07-18', 4100.25),
    (6, 'Frank Mueller', 'frank@example.com', 'Berlin', 'Germany', '2023-08-30', 1540.00),
    (7, 'Grace Chen', 'grace@example.com', 'Toronto', 'Canada', '2024-01-05', 2780.90),
    (8, 'Hassan Ali', 'hassan@example.com', 'Dubai', 'UAE', '2024-02-14', 5620.00),
    (9, 'Ingrid Larsson', 'ingrid@example.com', 'Stockholm', 'Sweden', '2024-03-20', 1120.30),
    (10, 'James Wright', 'james@example.com', 'Sydney', 'Australia', '2024-04-11', 890.00),
]
conn.executemany("INSERT INTO customers VALUES (?, ?, ?, ?, ?, ?, ?)", customers)

# Insert products (12 products across 4 categories)
products = [
    (1, 'Wireless Headphones', 'Electronics', 79.99, 150),
    (2, 'Mechanical Keyboard', 'Electronics', 129.99, 85),
    (3, 'USB-C Hub', 'Electronics', 49.99, 200),
    (4, 'Running Shoes', 'Sports', 119.99, 60),
    (5, 'Yoga Mat', 'Sports', 34.99, 120),
    (6, 'Water Bottle', 'Sports', 24.99, 300),
    (7, 'Python Cookbook', 'Books', 44.99, 90),
    (8, 'Data Science Handbook', 'Books', 39.99, 70),
    (9, 'AI Fundamentals', 'Books', 54.99, 45),
    (10, 'Standing Desk', 'Furniture', 349.99, 25),
    (11, 'Ergonomic Chair', 'Furniture', 449.99, 15),
    (12, 'Desk Lamp', 'Furniture', 59.99, 110),
]
conn.executemany("INSERT INTO products VALUES (?, ?, ?, ?, ?)", products)

# Insert orders (20 orders spanning 2024)
orders = [
    (1, 1, '2024-01-10', 209.98, 'completed'),
    (2, 2, '2024-01-25', 129.99, 'completed'),
    (3, 3, '2024-02-05', 449.99, 'completed'),
    (4, 5, '2024-02-18', 164.98, 'completed'),
    (5, 8, '2024-03-02', 799.98, 'completed'),
    (6, 1, '2024-03-15', 84.98, 'completed'),
    (7, 7, '2024-04-01', 259.98, 'completed'),
    (8, 3, '2024-04-20', 119.99, 'completed'),
    (9, 4, '2024-05-08', 79.99, 'completed'),
    (10, 6, '2024-05-22', 174.98, 'completed'),
    (11, 5, '2024-06-10', 349.99, 'completed'),
    (12, 8, '2024-06-28', 539.98, 'completed'),
    (13, 9, '2024-07-14', 94.98, 'completed'),
    (14, 2, '2024-07-30', 449.99, 'completed'),
    (15, 10, '2024-08-12', 59.98, 'completed'),
    (16, 7, '2024-09-05', 154.98, 'completed'),
    (17, 3, '2024-09-18', 79.99, 'completed'),
    (18, 1, '2024-10-03', 349.99, 'completed'),
    (19, 5, '2024-11-15', 129.99, 'shipped'),
    (20, 8, '2024-12-01', 259.97, 'shipped'),
]
conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", orders)

# Insert order items (line items for each order)
order_items = [
    (1, 1, 1, 1, 79.99),    # Order 1: Headphones
    (2, 1, 2, 1, 129.99),   # Order 1: Keyboard
    (3, 2, 2, 1, 129.99),   # Order 2: Keyboard
    (4, 3, 11, 1, 449.99),  # Order 3: Ergonomic Chair
    (5, 4, 4, 1, 119.99),   # Order 4: Running Shoes
    (6, 4, 7, 1, 44.99),    # Order 4: Python Cookbook
    (7, 5, 10, 1, 349.99),  # Order 5: Standing Desk
    (8, 5, 11, 1, 449.99),  # Order 5: Ergonomic Chair
    (9, 6, 5, 1, 34.99),    # Order 6: Yoga Mat
    (10, 6, 3, 1, 49.99),   # Order 6: USB-C Hub
    (11, 7, 2, 2, 129.99),  # Order 7: 2x Keyboard
    (12, 8, 4, 1, 119.99),  # Order 8: Running Shoes
    (13, 9, 1, 1, 79.99),   # Order 9: Headphones
    (14, 10, 8, 1, 39.99),  # Order 10: Data Science Handbook
    (15, 10, 9, 1, 54.99),  # Order 10: AI Fundamentals
    (16, 10, 1, 1, 79.99),  # Order 10: Headphones (Frank bought books + headphones)
    (17, 11, 10, 1, 349.99),# Order 11: Standing Desk
    (18, 12, 9, 2, 54.99),  # Order 12: 2x AI Fundamentals
    (19, 12, 11, 1, 449.99),# Order 12: Ergonomic Chair (Hassan's 2nd order, big spender)
    (20, 13, 5, 1, 34.99),  # Order 13: Yoga Mat
    (21, 13, 12, 1, 59.99), # Order 13: Desk Lamp
    (22, 14, 11, 1, 449.99),# Order 14: Ergonomic Chair
    (23, 15, 12, 1, 59.99), # Order 15: Desk Lamp (James from Australia, small order)
    (24, 16, 9, 1, 54.99),  # Order 16: AI Fundamentals
    (25, 16, 7, 1, 44.99),  # Order 16: Python Cookbook
    (26, 16, 9, 1, 54.99),  # Order 16: Another AI Fundamentals (Grace bought 2 books)
    (27, 17, 1, 1, 79.99),  # Order 17: Headphones
    (28, 18, 10, 1, 349.99),# Order 18: Standing Desk
    (29, 19, 2, 1, 129.99), # Order 19: Keyboard
    (30, 20, 3, 2, 49.99),  # Order 20: 2x USB-C Hub
    (31, 20, 12, 1, 59.99), # Order 20: Desk Lamp
    (32, 20, 6, 4, 24.99),  # Order 20: 4x Water Bottles (Hassan stocking up)
]
conn.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?, ?)", order_items)

conn.commit()
print("Database created successfully!")
print(f"  - {conn.execute('SELECT COUNT(*) FROM customers').fetchone()[0]} customers")
print(f"  - {conn.execute('SELECT COUNT(*) FROM products').fetchone()[0]} products")
print(f"  - {conn.execute('SELECT COUNT(*) FROM orders').fetchone()[0]} orders")
print(f"  - {conn.execute('SELECT COUNT(*) FROM order_items').fetchone()[0]} order items")

## Define Agent Tools

Our agent needs two tools to interact with the database:

1. **`inspect_schema`** - Retrieves the full database schema (CREATE TABLE statements and sample rows). The agent calls this first to understand what tables and columns exist.

2. **`validate_and_execute_sql`** - Takes a SQL query, validates it for safety (rejecting writes, enforcing row limits), executes it, and returns formatted results.

By separating schema inspection from query execution, we encourage the agent to always look at the schema before writing SQL. This reduces hallucinated column names significantly.

In [ ]:
def inspect_schema():
    """Return the full database schema with CREATE TABLE statements and sample data."""
    schema_parts = []

    # Get all table names
    tables = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()

    for (table_name,) in tables:
        # Get the CREATE TABLE statement
        create_stmt = conn.execute(
            "SELECT sql FROM sqlite_master WHERE type='table' AND name=?",
            (table_name,)
        ).fetchone()[0]
        schema_parts.append(create_stmt + ";")

        # Get sample rows (first 3)
        cursor = conn.execute(f"SELECT * FROM {table_name} LIMIT 3")
        columns = [desc[0] for desc in cursor.description]
        rows = cursor.fetchall()

        schema_parts.append(f"\n-- Sample data from {table_name}:")
        schema_parts.append("-- " + " | ".join(columns))
        for row in rows:
            schema_parts.append("-- " + " | ".join(str(v) for v in row))
        schema_parts.append("")

    # Get row counts
    schema_parts.append("-- Row counts:")
    for (table_name,) in tables:
        count = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        schema_parts.append(f"-- {table_name}: {count} rows")

    return "\n".join(schema_parts)


def validate_and_execute_sql(query):
    """Validate and execute a SQL query with safety checks."""
    # Safety: reject any write operations
    forbidden = ['INSERT', 'UPDATE', 'DELETE', 'DROP', 'ALTER', 'CREATE', 'TRUNCATE']
    query_upper = query.upper().strip()
    for keyword in forbidden:
        # Check that the keyword appears as a statement start or standalone word
        if keyword in query_upper.split():
            return f"Error: {keyword} operations are not allowed. This agent is read-only."

    # Add LIMIT if not present to prevent huge result sets
    if 'LIMIT' not in query_upper:
        query = query.rstrip(';') + f' LIMIT {MAX_ROW_LIMIT}'

    try:
        cursor = conn.execute(query)
        columns = [desc[0] for desc in cursor.description]
        rows = cursor.fetchall()

        # Format results as a readable table
        result = " | ".join(columns) + "\n"
        result += "-" * len(result) + "\n"
        for row in rows:
            result += " | ".join(str(v) for v in row) + "\n"
        result += f"\n({len(rows)} rows returned)"
        return result
    except Exception as e:
        return f"SQL Error: {str(e)}"


# Quick test: verify the tools work
print("=== Schema Preview (first 20 lines) ===")
print("\n".join(inspect_schema().split("\n")[:20]))
print("\n...\n")
print("=== Sample Query ===")
print(validate_and_execute_sql("SELECT name, country, lifetime_spend FROM customers ORDER BY lifetime_spend DESC LIMIT 5"))

## Tool Schemas for the Responses API

The OpenAI Responses API requires tool definitions as JSON schemas. Each tool schema specifies:
- The function name and description
- The parameters with their types
- `strict: True` to enforce exact parameter matching

In [ ]:
tools = [
    {
        "type": "function",
        "name": "inspect_schema",
        "description": (
            "Inspect the database schema. Returns all CREATE TABLE statements, "
            "sample data from each table, and row counts. Call this FIRST before "
            "writing any SQL queries so you know what tables and columns exist."
        ),
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "validate_and_execute_sql",
        "description": (
            "Validate and execute a SQL query against the database. "
            "The query must be a SELECT statement; write operations (INSERT, UPDATE, "
            "DELETE, DROP, etc.) are not allowed. A row LIMIT is automatically applied "
            "if not present. Returns the results as a formatted table."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The SQL SELECT query to execute."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        },
        "strict": True
    }
]

print("Tools defined:")
for tool in tools:
    print(f"  - {tool['name']}: {tool['description'][:80]}...")

## Build the Agent Loop

The agent loop is the core of our system. Here is how it works:

1. The user's question is sent to the model along with the tool definitions
2. If the model responds with tool calls, we execute them and feed the results back
3. This continues until the model produces a final text response (no more tool calls)
4. We cap the loop at 5 iterations as a safety measure

The system instructions tell the model to always inspect the schema first, write clean SQL, and provide clear explanations.

In [ ]:
SYSTEM_INSTRUCTIONS = """You are a data analysis agent with access to a SQLite database.

Your workflow:
1. First, inspect the schema to understand available tables and columns
2. Write a SQL query to answer the user's question
3. Execute the query using the validate_and_execute_sql tool
4. Interpret the results and provide a clear, plain-English answer

Rules:
- Always inspect the schema first if you haven't already
- Write clean, efficient SQL
- Explain your reasoning
- Include relevant numbers and insights in your answer
- If the query returns unexpected results, explain why
- Format numbers nicely (e.g., $1,234.56 for currency)"""


def ask_data_question(question):
    """
    Agent loop: takes a natural language question, uses tools to inspect
    schema and run SQL, then provides a plain-English answer.
    """
    messages = [{"role": "user", "content": question}]

    print(f"Question: {question}")
    print("=" * 60)

    for turn in range(5):  # Max 5 tool-calling turns
        response = client.responses.create(
            model=MODEL,
            instructions=SYSTEM_INSTRUCTIONS,
            input=messages,
            tools=tools
        )

        # Check for tool calls in the response
        tool_calls = [item for item in response.output if item.type == "function_call"]

        # If no tool calls, the model produced a final answer
        if not tool_calls:
            answer = response.output_text
            print(f"\nAnswer:\n{answer}")
            return answer

        # Process each tool call
        for tool_call in tool_calls:
            args = json.loads(tool_call.arguments)

            if tool_call.name == "inspect_schema":
                print(f"\n[Turn {turn + 1}] Inspecting database schema...")
                result = inspect_schema()
            elif tool_call.name == "validate_and_execute_sql":
                print(f"\n[Turn {turn + 1}] Executing SQL:")
                print(f"  {args['query']}")
                result = validate_and_execute_sql(args["query"])
                print(f"\n[Results]\n{result}")
            else:
                result = f"Unknown tool: {tool_call.name}"

            # Append the tool call and its output to the conversation
            messages.append(tool_call)
            messages.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": result
            })

    return "Max turns reached without a final answer."


print("Agent loop defined. Ready to answer questions!")

## Test the Agent with Analytical Questions

Let's put the agent through its paces with a variety of SQL question types: aggregations, joins, date analysis, grouping, and subqueries.

### Question 1: Aggregation
A straightforward ranking question that requires summing order totals.

In [ ]:
ask_data_question("What are the top 5 customers by total spending?")

### Question 2: Joins + Aggregation
This requires joining `order_items` with `products` and grouping by category.

In [ ]:
ask_data_question("Which product category generates the most revenue?")

### Question 3: Date Analysis
The agent needs to extract month from `order_date` and count orders per month.

In [ ]:
ask_data_question("How many orders were placed per month in 2024?")

### Question 4: Grouping by Country
Requires joining customers with orders and computing averages per country.

In [ ]:
ask_data_question("What is the average order value by country?")

### Question 5: LEFT JOIN + NULL Check
The agent needs to find products with no matching entries in `order_items`.

In [ ]:
ask_data_question("Which products have never been ordered?")

### Question 6: Subquery
Requires computing the average first, then filtering customers above it.

In [ ]:
ask_data_question("Show me customers who spent more than the average")

## Safety Demonstration

Let's verify that our safety guardrails work correctly. The agent should refuse to execute any write operations, even if the user explicitly asks for them.

In [ ]:
# This should be rejected by the safety checks
ask_data_question("Delete all customers from the database")

In [ ]:
# Test the safety check directly to confirm it works
print("Direct safety check tests:")
print()
print("1. DELETE:", validate_and_execute_sql("DELETE FROM customers WHERE id = 1"))
print()
print("2. DROP:", validate_and_execute_sql("DROP TABLE customers"))
print()
print("3. UPDATE:", validate_and_execute_sql("UPDATE customers SET name = 'hacked' WHERE id = 1"))
print()
print("4. Valid SELECT (should work):")
print(validate_and_execute_sql("SELECT COUNT(*) as total_customers FROM customers"))

## Summary

In this notebook, we built a complete SQL Data Analysis Agent that converts natural language questions into database queries. Here are the key takeaways:

- **Text-to-SQL agents** convert natural language into database queries, making data accessible to non-technical users
- **Schema inspection** is a critical first step -- by giving the model access to the real schema, we dramatically reduce hallucinated table and column names
- **SQL validation is essential** -- always reject write operations (`DROP`, `DELETE`, `UPDATE`, etc.) before they reach the database
- **Row limits** prevent memory issues and protect against queries that return enormous result sets
- **The model can interpret results** and provide business insights, going beyond raw numbers to explain what the data means
- **This pattern generalizes** to any SQL database (PostgreSQL, MySQL, etc.) -- just swap the connection and adjust the schema inspection logic